# 01. Exploratory Data Analysis (EDA)
## Fan Predictive Maintenance System

This notebook analyzes raw sensor telemetry collected from the ESP32 sensor platform (MPU6050 + DHT22 + Current/Voltage sensors).

### Objectives:
1. Inspect data schema and check for missing values
2. Compare Healthy Baseline distributions vs. Anomaly conditions
3. Analyze sensor correlations and multi-sensor fault signatures

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

### 1. Load Dataset

In [ ]:
data_path = '../data/predictive_maintenance_dataset.csv'
df = pd.read_csv(data_path)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
print(f"Total Samples: {len(df)}")
df.head()

### 2. Dataset Summary & Missing Values

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Data Types & Info ===")
df.info()

### 3. Distribution of Machine Conditions and Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Status'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0].set_title('Operational Status Breakdown')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['Condition'].value_counts().plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Fault Condition Frequency')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

### 4. Healthy Baseline vs. Anomaly Statistics

In [ ]:
features = ['TotalVibration', 'Temp', 'Voltage', 'Current']
print("=== HEALTHY BASELINE STATISTICS ===")
healthy_df = df[df['Status'] == 'Normal']
display(healthy_df[features].describe().T[['mean', 'std', 'min', '50%', 'max']])

print("\n=== FAULT CONDITIONS STATISTICS (MEAN) ===")
display(df.groupby('Condition')[features].mean())

### 5. Sensor Distributions & Box Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.boxplot(data=df, x='Condition', y=feat, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{feat} by Condition')
    axes[i].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

### 6. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Sensor Correlation Matrix')
plt.show()